In [8]:
%pip install -q "anthropic>=2.8.0,<3" pillow pendulum python-dotenv openai

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.2.17 requires numpy<2,>=1; python_version < "3.12", but you have numpy 2.0.2 which is incompatible.
langchain-community 0.2.16 requires numpy<2,>=1; python_version < "3.12", but you have numpy 2.0.2 which is incompatible.
google-genai 1.38.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.
deepeval 1.1.6 requires grpcio~=1.63.0, but you have grpcio 1.75.0 which is incompatible.
arxiv 2.1.3 requires requests~=2.32.0, but you have requests 2.34.2 which is incompatible.

[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# 1. Chiamata ad API claude zero-shot approach 

In [1]:
import io
import os
from PIL import Image
import base64
from anthropic import Anthropic
import json
from pathlib import Path
import pendulum

def resize_image_under_limit(image_path, max_size_mb=4.5):
    """
    Ridimensiona un'immagine se supera il limite di megabyte specificato,
    salvando temporaneamente l'immagine ottimizzata in memoria.
    """
    max_size_bytes = max_size_mb * 1024 * 1024
    img = Image.open(image_path)

    # Determina il formato originale
    img_format = img.format if img.format else 'JPEG'
    if img_format == 'MPO':
        img_format = 'JPEG'

    # Controlla la dimensione iniziale del file
    file_size = os.path.getsize(image_path)
    print(f"Dimensione originale dell'immagine: {file_size / (1024*1024):.2f} MB")

    if file_size <= max_size_bytes:
        print("L'immagine è già sotto il limite consentito.")
        with open(image_path, "rb") as f:
            return f.read(), img_format.lower()

    # Se supera il limite, riduciamo la risoluzione progressivamente
    quality = 90
    scale_factor = 0.9

    while file_size > max_size_bytes and scale_factor > 0.1:
        width, height = img.size
        new_size = (int(width * scale_factor), int(height * scale_factor))
        temp_img = img.resize(new_size, Image.Resampling.LANCZOS)

        buffer = io.BytesIO()
        temp_img.save(buffer, format=img_format, quality=quality)
        data = buffer.getvalue()
        file_size = len(data)

        print(f"Ridimensionamento... Nuova dimensione: {file_size / (1024*1024):.2f} MB (Scala: {scale_factor:.1f})")
        scale_factor -= 0.1
        quality = max(50, quality - 5)

    return data, img_format.lower()

In [5]:
# Inserisci ANTHROPIC_API_KEY nel file .env locale (vedi .env.example).
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")
api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    raise RuntimeError("Imposta ANTHROPIC_API_KEY nel file .env nella cartella di lavoro.")
client = Anthropic(api_key=api_key)

def analyze_image_with_claude(image_path="C:\\Users\\crosi\\Documents\\GitHub\\DH27\\images", prompt="Please analyze the attached image and describe it according to the instructions. Return only the JSON array.", system_prompt="""
### ROLE
You are an expert in medieval diagram analysis. Your task is to analyse the diagram in the image in a single pass: classify the diagram by subject, describe the relationships between its elements on the basis of what is visible, and interpret them using your knowledge of medieval iconography, texts and diagrammatic traditions.
 
### GOAL
Your goal is to classify the diagram by subject (e.g. cosmological, computistical) and provide an interpretation.
 
### INTERPRETATION RULES
- Base every interpretation on the visible features and the transcribed text first; use external knowledge only to complete them. State in "basis" what the interpretation mainly relies on.
- Do not invent sources. Mention an author, text or tradition only if you are confident of it; otherwise give the interpretation without naming a source and lower "interpretation_certainty".
- If an element or relation cannot be interpreted, write "uninterpreted" in the corresponding field and set "interpretation_certainty" to "low".
- When the same element appears in several entries, give it the same interpretation every time.
### OUTPUT
Return only a valid JSON object, with no explanation, no additional text, and no markdown code fences. Any double quote inside a string value must be escaped (\").
 
### OUTPUT FORMAT
[
  {
    "id": 1,
    "diagram_classification": "",
    "interpretation": "",
    "basis": "textual",
    "certainty": "high",
    "evidence": "",
    "updated_at": ""
  }
]
 
### FIELD RULES
- "id", "subject", "predicate", "object": copied unchanged from Step 1.
- "classification": the category of the diagram (e.g. cosmological, theological...).
- "interpretation": interpretation of the meaning of the diagram.
- "basis": the main ground of the interpretation, exactly one of "visual" (visible features), "textual" (transcribed text in the diagram), "external" (knowledge not present in the image).
- "certainty": your confidence in the interpretation of that entry: "high", "medium", or "low".
- "evidence": the reasons supporting the interpretation, distinguishing what is visible or written in the diagram from what comes from external knowledge.
- "updated_at": always an empty string; it will be filled in by a Python library.
"""):
    # Ridimensiona l'immagine
    image_bytes, img_type = resize_image_under_limit(image_path)

    # Se il formato è jpg, Anthropic richiede 'jpeg'
    media_type = f"image/{'jpeg' if img_type == 'jpg' else img_type}"

    # Codifica in Base64
    base64_image = base64.b64encode(image_bytes).decode('utf-8')

    # Chiamata API a Claude 3.5 Sonnet impostando sia il system prompt che l'user prompt
    message = client.messages.create(
        model="claude-sonnet-5-5",
        max_tokens=4096,
        system=system_prompt,  # Qui definiamo il system prompt
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": media_type,
                            "data": base64_image,
                        },
                    },
                    {
                        "type": "text",
                        "text": prompt
                    }
                ],
            }
        ],
    )
    return message.content[0].text



project_dir = Path(r"C:\Users\crosi\Documents\GitHub\DH27")
image_path = project_dir / "images" / "1_image_0001.jpg"
output_path = project_dir / "image_analysis_interpretation.json"

response = analyze_image_with_claude(str(image_path))
data = json.loads(response)
if not isinstance(data, list):
    raise ValueError("Expected a JSON array from Claude.")
if not all(isinstance(item, dict) for item in data):
    raise ValueError("Expected every JSON array entry to be an object.")

updated_at = pendulum.now("UTC").format("YYYY-MM-DDTHH:mm[Z]")
for item in data:
    item["updated_at"] = updated_at

with output_path.open("w", encoding="utf-8") as file:
    json.dump(data, file, ensure_ascii=False, indent=2)
print(f"Saved to: {output_path}")

Dimensione originale dell'immagine: 0.06 MB
L'immagine è già sotto il limite consentito.
Saved to: C:\Users\crosi\Documents\GitHub\DH27\image_analysis_interpretation.json


In [8]:
print(response)

[
  {
    "id": 1,
    "relation_type": "text_shape",
    "subject": "PATER",
    "predicate": "inside",
    "object": "top-left circle at the upper-left vertex of the triangle",
    "certainty": "high",
    "evidence": "The word is written in the roundel at the upper-left corner of the inverted triangle.",
    "updated_at": ""
  },
  {
    "id": 2,
    "relation_type": "text_shape",
    "subject": "SPS SCS",
    "predicate": "inside",
    "object": "top-right circle at the upper-right vertex of the triangle",
    "certainty": "low",
    "evidence": "The text in the upper-right roundel is partly abraded and its reading is uncertain. It is probably an abbreviation for Spiritus Sanctus.",
    "updated_at": ""
  },
  {
    "id": 3,
    "relation_type": "text_shape",
    "subject": "DEUS",
    "predicate": "inside",
    "object": "central circle",
    "certainty": "medium",
    "evidence": "The word is written inside the double-ringed circle at the center of the triangle. The letters are s

# 2. Casistica in cui l'interpretazione viene fornita dopo un primo riconoscimento delle forme (CLAUDE) - Chain of Thought (CoT)

In [5]:

class ClaudeChat:
    def __init__(self, system_prompt=""):
        import os
        from inspect import signature
        from pathlib import Path
        from anthropic import Anthropic
        from dotenv import load_dotenv

        load_dotenv(Path.cwd() / ".env")
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("Imposta ANTHROPIC_API_KEY nel file .env nella cartella di lavoro.")

        self.client = Anthropic(api_key=api_key)
        if "cache_control" not in signature(self.client.messages.create).parameters:
            raise RuntimeError("Aggiorna anthropic eseguendo la prima cella, poi riavvia il kernel e riesegui il notebook.")
        self.system_prompt = system_prompt
        self.messages = []

    def start_chat_with_image(self, image_path, prompt="Please analyze the attached image and describe it according to the instructions. Return only the JSON array."):
        """
        Inizia la conversazione inviando l'immagine ridimensionata e il primo prompt.
        """
        # Ridimensiona e prepara l'immagine
        image_bytes, img_type = resize_image_under_limit(image_path)
        media_type = f"image/{'jpeg' if img_type == 'jpg' else img_type}"
        base64_image = base64.b64encode(image_bytes).decode('utf-8')

        # Crea il primo messaggio con l'immagine e il testo
        self.messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": media_type,
                            "data": base64_image,
                        },
                    },
                    {
                        "type": "text",
                        "text": prompt
                    }
                ],
            }
        ]

        return self._send_request()

    def send_message(self, prompt):
        """
        Invia un messaggio di testo successivo mantenendo la cronologia.
        """
        if not self.messages:
            raise ValueError("Inizia prima la conversazione usando 'start_chat_with_image'.")

        # Aggiunge il nuovo messaggio dell'utente alla cronologia
        self.messages.append({
            "role": "user",
            "content": [{"type": "text", "text": prompt}]
        })

        return self._send_request()

    def _send_request(self):
        """
        Invia l'intera cronologia dei messaggi a Claude.
        """
        response = self.client.messages.create(
            model="claude-sonnet-5-5",
            max_tokens=4096,
            system=self.system_prompt,
            cache_control={"type": "ephemeral"},
            messages=self.messages
        )
        self.last_usage = response.usage

        assistant_response = "".join(
            block.text for block in response.content
            if getattr(block, "type", None) == "text" and block.text
        )
        if not assistant_response:
            tipi = [getattr(b, "type", None) for b in response.content]
            raise RuntimeError(f"Nessun testo nella risposta (stop_reason={response.stop_reason}, blocchi={tipi})")

        # Aggiunge la risposta dell'assistente alla cronologia per i turni successivi
        self.messages.append({
            "role": "assistant",
            "content": [{"type": "text", "text": assistant_response}]
        })

        return assistant_response

In [6]:
# Esempio interattivo: l'immagine rimane nel contesto!
# Sostituisci con il percorso di una tua immagine reale per testare
percorso_immagine = 'C:\\Users\\crosi\\Documents\\GitHub\\DH27\\images\\1_image_0001.jpg'
basic_prompt = (Path.cwd() / "systemprompt_general.md").read_text(encoding="utf-8")
# Istruzioni comuni ai due turni
chat_session = ClaudeChat(system_prompt=basic_prompt)

interpretation_prompt = (Path.cwd() / "userprompt_interpretation.md").read_text(encoding="utf-8")

element_prompt = (Path.cwd() / "userprompt_basic.md").read_text(encoding="utf-8")
try:
    # 2. Primo turno: analisi element-level dell'immagine
    print("--- Turno 1 (Invio Immagine) ---")
    risposta_1 = chat_session.start_chat_with_image(percorso_immagine, prompt=element_prompt)
    print(f"Claude: {risposta_1}\n")
    element_analysis = json.loads(risposta_1)
    if not isinstance(element_analysis, list) or not all(isinstance(item, dict) for item in element_analysis):
        raise ValueError("Il primo turno deve restituire un array JSON di oggetti.")
    print(f"Token scritti nella cache: {chat_session.last_usage.cache_creation_input_tokens}")

    # 3. Secondo turno: facciamo una domanda specifica sull'immagine SENZA inviarla di nuovo
    print("--- Turno 2 (Solo testo, l'immagine è ancora nel contesto) ---")
    risposta_2 = chat_session.send_message(interpretation_prompt)
    print(f"Claude: {risposta_2}")
    interpretation = json.loads(risposta_2)
    if not isinstance(interpretation, list) or not all(isinstance(item, dict) for item in interpretation):
        raise ValueError("Il secondo turno deve restituire un array JSON di oggetti.")
    print(f"Token letti dalla cache: {chat_session.last_usage.cache_read_input_tokens}")

    output_path = Path.cwd() / "two_step_analysis.json"
    with output_path.open("w", encoding="utf-8") as file:
        json.dump({"element_analysis": element_analysis, "interpretation": interpretation}, file, ensure_ascii=False, indent=2)
    print(f"Salvato in: {output_path}")
except Exception as e:
    print(f"Errore {type(e).__name__}: {e}")
    raise

--- Turno 1 (Invio Immagine) ---
Dimensione originale dell'immagine: 0.06 MB
L'immagine è già sotto il limite consentito.
Errore APIConnectionError: Connection error.


APIConnectionError: Connection error.

# 3. Approccio Zero-Shot con OpenAI (GPT)

In [ ]:
%%bash
pip install -q openai

In [ ]:
import io
import os
import base64
from openai import OpenAI
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")
openai_key = os.environ.get("OPENAI_API_KEY")
if not openai_key:
    raise RuntimeError("Imposta OPENAI_API_KEY nel file .env nella cartella di lavoro.")
openai_client = OpenAI(api_key=openai_key)

def analyze_image_with_gpt(image_path, prompt="Descrivi questa immagine in dettaglio.", system_prompt="Sei un assistente esperto di analisi visiva. Rispondi in modo chiaro e conciso in italiano."):
    # Riutilizziamo la funzione precedente per ridimensionare l'immagine sotto il limite
    image_bytes, img_type = resize_image_under_limit(image_path)

    media_type = f"image/{'jpeg' if img_type == 'jpg' else img_type}"
    base64_image = base64.b64encode(image_bytes).decode('utf-8')

    # Chiamata API a GPT-4o
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{media_type};base64,{base64_image}"
                        }
                    }
                ]
            }
        ],
        max_tokens=1024
    )
    return response.choices[0].message.content

# 4. Approccio CoT GPT

In [ ]:
class GPTChat:
    def __init__(self, system_prompt="Sei un assistente esperto in italiano."):
        self.client = openai_client
        self.system_prompt = system_prompt
        # Inizializziamo i messaggi inserendo subito il system prompt
        self.messages = [
            {
                "role": "system",
                "content": self.system_prompt
            }
        ]

    def start_chat_with_image(self, image_path, prompt="Analizza questa immagine"):
        """
        Inizia la conversazione inviando l'immagine ridimensionata e il primo prompt.
        """
        image_bytes, img_type = resize_image_under_limit(image_path)
        media_type = f"image/{'jpeg' if img_type == 'jpg' else img_type}"
        base64_image = base64.b64encode(image_bytes).decode('utf-8')

        # Aggiungiamo il primo messaggio dell'utente con l'immagine e il testo
        self.messages.append({
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:{media_type};base64,{base64_image}"
                    }
                }
            ]
        })

        return self._send_request()

    def send_message(self, prompt):
        """
        Invia un messaggio di testo successivo mantenendo la cronologia.
        """
        if len(self.messages) <= 1:
            raise ValueError("Inizia prima la conversazione usando 'start_chat_with_image'.")

        self.messages.append({
            "role": "user",
            "content": prompt
        })

        return self._send_request()

    def _send_request(self):
        """
        Invia l'intera cronologia dei messaggi a GPT.
        """
        response = self.client.chat.completions.create(
            model="gpt-4o",
            messages=self.messages,
            max_tokens=1024
        )

        assistant_response = response.choices[0].message.content

        # Aggiunge la risposta dell'assistente alla cronologia per mantenere il contesto
        self.messages.append({
            "role": "assistant",
            "content": assistant_response
        })

        return assistant_response

In [ ]:
# Esempio interattivo con GPT:
# percorso_immagine_gpt = 'percorso/della/tua/immagine.png'

# gpt_chat_session = GPTChat(system_prompt="Sei uno scienziato attento ai dettagli.")

try:
    # Turno 1 con immagine
    # risposta_1 = gpt_chat_session.start_chat_with_image(percorso_immagine_gpt, prompt="Descrivi cosa vedi in questa immagine.")
    # print("GPT Turno 1:", risposta_1)

    # Turno 2 (l'immagine rimane nel contesto)
    # risposta_2 = gpt_chat_session.send_message("Quali potrebbero essere le implicazioni scientifiche di ciò che hai appena descritto?")
    # print("GPT Turno 2:", risposta_2)
except Exception as e:
    pass